## 🌍 *Tutorial #3*: **Using PINNs to simulate the Air Pollution Dynamics**

In the previous tutorial, we used a PINN that combined **measurement data** with the **governing physics equation**. In this tutorial, we now take the idea one step further and train the PINN **without any measurement data at all**.

Instead, we use the PINN to simulate the concentration dynamics purely from:

- the governing differential equation,
- the initial condition,
- and the boundary conditions.

In other words, we use the PINN as a physics-based solver that reconstructs the full spatio-temporal evolution of the system from first principles. It targets the question:

> Can the model predict the pollution dynamics even **without any measurement data**?

--- 

### ⚙️ Problem Setup 

We assume that we know the initial spread of the pollutant (at $t=0$) and that the concentration at both boundaries remains zero for all later times.

This provides all information required to uniquely define the **forward problem**, allowing the PINN to simulate the concentration dynamics from scratch—similar to what a classical numerical PDE solver would do.

---

### 🌊 Governing Equations

- **PDE**: 
$
\begin{equation}
\frac{\partial C}{\partial t} = 0.01 \frac{\partial C^2}{\partial^2 x},
\end{equation}
$

- **Initial Condition (IC)**:
$
\begin{equation}
C(x, t=0)= \exp\left(\frac{-(x - 0.4)^2}{0.01}\right), \quad 0\le x\le 1
\end{equation}
$

- **Boundary Conditions (BC)**:
$
\begin{equation}
C(x=0, t)= C(x=1, t) = 0, \quad t>0
\end{equation}
$

---

### 📜 How PINNs Incorporate IC and BC

PINNs can enforce initial and boundary conditions in two common ways:

#### 1️⃣ Soft constraints (used here)

The IC and BC are provided as **labeled training points** and included through additional loss terms.

This means the PINN minimizes:

- PDE residual loss
- Initial condition loss
- Boundary condition loss

#### 2️⃣ Hard constraints (not covered here)

The network **architecture is modified** so that the solution automatically satisfies the IC and BC by construction.

This can make training fully unsupervised, since no labeled data points are required.

---

### ➡️ Our Approach

In this tutorial, we use the **soft-constraint formulation**, since it is easier to implement and clearly illustrates how PINNs combine different physical constraints during training.

Let us begin by implementing the functions that define the initial and boundary conditions.

In [ ]:
import numpy as np
import torch

# --------------------------------------------------
# Initial Condition: C(x,0)
# --------------------------------------------------
def initial_condition(x):
    """
    Gaussian pollutant plume at t = 0

    Parameters
    ----------
    x : numpy array
        Spatial coordinates

    Returns
    -------
    C0 : numpy array
        Initial concentration values
    """
    return np.exp(-((x - 0.4) ** 2) / 0.01)


# --------------------------------------------------
# Boundary Conditions:
# C(0,t) = 0 and C(1,t) = 0
# --------------------------------------------------
def boundary_condition(t):
    """
    Zero concentration at both boundaries.

    Parameters
    ----------
    t : numpy array
        Time coordinates

    Returns
    -------
    numpy array
        Zero values
    """
    return np.zeros_like(t)


# --------------------------------------------------
# Sampling IC + BC training points
# --------------------------------------------------
def sample_ic_bc_points(N_ic=100, N_bc=100):
    """
    Sample points for PINN training:
    - Initial condition points
    - Boundary condition points

    Parameters
    ----------
    N_ic : int
        Number of initial condition points
    N_bc : int
        Number of boundary points (per boundary)

    Returns
    -------
    dict containing tensors
    """

    # -------------------------
    # Initial condition points
    # -------------------------
    x_ic = np.linspace(0, 1, N_ic)
    t_ic = np.zeros_like(x_ic)

    C_ic = initial_condition(x_ic)

    X_ic = np.column_stack([x_ic, t_ic])

    # -------------------------
    # Boundary x = 0
    # -------------------------
    t_bc0 = np.linspace(0, 1, N_bc)
    x_bc0 = np.zeros_like(t_bc0)

    C_bc0 = boundary_condition(t_bc0)

    X_bc0 = np.column_stack([x_bc0, t_bc0])

    # -------------------------
    # Boundary x = 1
    # -------------------------
    t_bc1 = np.linspace(0, 1, N_bc)
    x_bc1 = np.ones_like(t_bc1)

    C_bc1 = boundary_condition(t_bc1)

    X_bc1 = np.column_stack([x_bc1, t_bc1])

    # -------------------------
    # Convert to tensors
    # -------------------------
    data = {
        "X_ic": torch.tensor(X_ic, dtype=torch.float32),
        "y_ic": torch.tensor(C_ic.reshape(-1,1), dtype=torch.float32),

        "X_bc0": torch.tensor(X_bc0, dtype=torch.float32),
        "y_bc0": torch.tensor(C_bc0.reshape(-1,1), dtype=torch.float32),

        "X_bc1": torch.tensor(X_bc1, dtype=torch.float32),
        "y_bc1": torch.tensor(C_bc1.reshape(-1,1), dtype=torch.float32),
    }

    return data

### 📍 Visualizing the Sampled Training Points

Before training the PINN, it is always a good idea to inspect the sampled points used during optimization.

Recall that we now work with three different types of points:

- **Collocation points** inside the domain, where the diffusion equation is enforced (*we provide a predefined function from the previous tutorial for this*),
- **Initial condition points** at $t=0$, where the initial pollutant distribution is prescribed,
- **Boundary condition points** at $x=0$ and $x=1$, where the concentration is fixed to zero.

Visualizing these points helps us verify that the sampling has been implemented correctly and that the domain is adequately covered.

In [ ]:
import matplotlib.pyplot as plt
from methods.model import sample_collocation_points

X_col = sample_collocation_points(N_col=1000)
data = sample_ic_bc_points(N_ic=100, N_bc=100)

plt.figure(figsize=(8, 6))

# -----------------------------------
# Collocation points (interior)
# -----------------------------------
plt.scatter(
    X_col[:, 1].detach().numpy(),   # time
    X_col[:, 0].detach().numpy(),   # space
    s=5,
    alpha=0.3,
    label="Collocation Points"
)

# -----------------------------------
# Initial condition points
# -----------------------------------
plt.scatter(
    data["X_ic"][:, 1].numpy(),   # time
    data["X_ic"][:, 0].numpy(),   # space
    s=40,
    marker="o",
    label="Initial Condition"
)

# -----------------------------------
# Boundary x = 0
# -----------------------------------
plt.scatter(
    data["X_bc0"][:, 1].numpy(),
    data["X_bc0"][:, 0].numpy(),
    s=40,
    marker="s",
    label="Boundary x=0"
)

# -----------------------------------
# Boundary x = 1
# -----------------------------------
plt.scatter(
    data["X_bc1"][:, 1].numpy(),
    data["X_bc1"][:, 0].numpy(),
    s=40,
    marker="s",
    label="Boundary x=1"
)

plt.xlabel("Time t")
plt.ylabel("Space x")
plt.title("Sampled PINN Training Points")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.2)
plt.show()

---

## 🔧 PDE Residuals, Physics Loss and Training Loop

In particular, the core components remain the same:

- the function that computes the PDE residual using automatic differentiation,
- the physics loss, which penalizes violations of the diffusion equation,
- and the overall training loop used to optimize the neural network parameters.

The main difference is that we now additionally include the initial condition and boundary condition losses, since no measurement data is available. These constraints replace the observational data used in earlier tutorials and provide the information required to solve the forward problem.

With this setup, the PINN learns the concentration field by simultaneously satisfying:

- the governing diffusion equation,
- the prescribed initial concentration profile,
- and the boundary values over time.

In [ ]:
def compute_pde_residual(model, X_col, D=0.01):
    """
    Computes the PDE residual for the diffusion equation.
    """

    x = X_col[:, 0:1]
    t = X_col[:, 1:2]

    # forward pass
    C = model(torch.cat([x, t], dim=1))

    # First-order derivatives
    C_t = torch.autograd.grad(
        C, t,
        grad_outputs=torch.ones_like(C),
        retain_graph=True,
        create_graph=True
    )[0]

    C_x = torch.autograd.grad(
        C, x,
        grad_outputs=torch.ones_like(C),
        retain_graph=True,
        create_graph=True
    )[0]

    # Second-order derivative
    C_xx = torch.autograd.grad(
        C_x, x,
        grad_outputs=torch.ones_like(C_x),
        retain_graph=True,
        create_graph=True
    )[0]

    # Diffusion residual
    residual = C_t - D * C_xx

    return residual

def MO_loss(model, X_icbc, y_icbc, X_col, lambda_phys=1.0):

    # -------------------------
    # Data loss
    # -------------------------
    C_pred = model(X_icbc)
    loss_data = torch.mean((C_pred - y_icbc) ** 2)

    # -------------------------
    # Physics loss
    # -------------------------
    residual = compute_pde_residual(model, X_col)
    loss_phys = torch.mean(residual ** 2)

    # Total loss
    loss = loss_data + lambda_phys * loss_phys

    return loss, loss_data, loss_phys


def train_pinn(model, optimizer,
               N_ic=128, N_bc=128,
               N_col=128,
               n_epochs=5000,
               lambda_phys=1.0,
               print_every=100):

    history = {
        "epoch": [],
        "loss": [],
        "loss_icbc": [],
        "loss_phys": [],
    }

    for epoch in range(1, n_epochs + 1):

        # -------------------------
        # Training step
        # -------------------------
        model.train()
        optimizer.zero_grad()

        # We sample IC/BC and collocation points anew at each epoch
        X_col = sample_collocation_points(N_col=N_col)
        data_icbc = sample_ic_bc_points(N_ic=N_ic, N_bc=N_bc)

        # -----------------------------------
        # Combine all input coordinates
        # -----------------------------------
        X_icbc = torch.cat(
            [
                data_icbc["X_ic"],
                data_icbc["X_bc0"],
                data_icbc["X_bc1"]
            ],
            dim=0
        )

        # -----------------------------------
        # Combine all target values
        # -----------------------------------
        y_icbc = torch.cat(
            [
                data_icbc["y_ic"],
                data_icbc["y_bc0"],
                data_icbc["y_bc1"]
            ],
            dim=0
        )

        # Multi-obejctive loss is evaluated
        loss, loss_icbc, loss_p = MO_loss(
            model, X_icbc, y_icbc, X_col, lambda_phys
        )

        loss.backward()
        optimizer.step()

        # -------------------------
        # Log history and print current losses
        # -------------------------
        history["epoch"].append(epoch)
        history["loss"].append(loss.item())
        history["loss_icbc"].append(loss_icbc.item())
        history["loss_phys"].append(loss_p.item())

        if epoch % print_every == 0 or epoch == 1:
            print(
                f"Epoch {epoch:5d} | "
                f"Total: {loss.item():1.2e} | "
                f"ICBC: {loss_icbc.item():1.2e} | "
                f"PDE: {loss_p.item():1.2e} | "
            )

    return history

---

## ▶️ PINN Training

We also use the same neural network architecture as in the previous tutorials in order to maintain a fair comparison.

This allows us to attribute differences in performance primarily to the training strategy and the additional physical constraints, rather than to changes in model capacity or architecture.

In other words, the network structure remains unchanged—the key difference is how the model is trained. Instead of relying on measurement data, the network now learns from the governing equation together with the prescribed initial and boundary conditions.

In [ ]:
from methods.model import NeuralNet

model = NeuralNet()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

N_col = 1024
N_ic = 512
N_bc = 128
n_epochs = 10000
lambda_phys = 0.1

history = train_pinn(
    model=model,
    optimizer=optimizer,
    N_ic=N_ic,
    N_bc=N_bc,
    N_col=N_col,
    n_epochs=n_epochs,
    lambda_phys=lambda_phys,
    print_every=200
)

A few comments on the chosen settings above:

- **We considerably increased the number of training epochs.**
This is because the forward problem is significantly harder to solve. In contrast to the previous tutorials, we do not have any measurement data inside the computational domain that would directly guide the model. The network must reconstruct the full dynamics solely from the information imposed at the initial time and at the boundaries, and then propagate this information throughout the domain.

- **We now have several training design choices to make.**
In particular, we must decide how many training points to use for:
    - the initial condition,
    - the boundary conditions,
    - and the collocation points.

An important remark in this context is that the **initial condition is the most critical component** of the learning task. If the model fails to learn the initial pollutant distribution accurately, recovering the true solution becomes very difficult. We therefore need to ensure that the initial condition is sampled sufficiently well and learned properly.

- **We reduced the weighting parameter $\lambda$.**
This is motivated by the same reasoning as above: in this setting, it is especially important that the network learns the initial condition accurately. Reducing $\lambda$ places relatively more emphasis on fitting the prescribed condition data, while still enforcing the PDE through the physics loss.

---

Let us now continue by inspecting the learning curves to see whether the initial and boundary conditions have been learned successfully and whether the PDE residuals have been minimized as well.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))

plt.plot(history["epoch"], history["loss_icbc"], label="ICBC Loss")
plt.plot(history["epoch"], history["loss_phys"], label="Physics Loss")

plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training History")
plt.legend()
plt.show()

We should observe that both loss terms — the **IC/BC** loss and the **physics loss** — decrease throughout the training process. This is a strong indication that the network is simultaneously learning to satisfy the prescribed initial and boundary conditions while also reducing violations of the governing diffusion equation.

We may also notice that the losses are still trending downward toward the end of training. This suggests that the model could likely benefit from additional optimization steps and has not yet fully converged.

At this stage, we have two reasonable options:

- **Increase the number of training epochs** and continue optimization to further refine the solution,
- or, for the sake of time, proceed with the current model and evaluate the results already obtained.

In practice, the appropriate choice depends on the desired accuracy and available computational budget.

---

If we decide to continue with the current model, let us now put it to the final test and compare its predictions once again with the reference solution.

In [ ]:
from methods.plots import plot_reference
plot_reference(model)

Well, if we have made suitable design choices, we should observe a relative $L_2$ error **close to 1%**. This is an excellent result and should be comparable to—or even better than—the previous models we tested.

What makes this particularly remarkable is that we achieved this **without using any measurement data** inside the domain. The model learned the full concentration dynamics purely from:

- the governing diffusion equation,
- the initial pollutant distribution,
- and the boundary conditions.

In other words, the PINN successfully acted as a **neural PDE solver**.

--- 

### 🔁 Further Experiments

You are encouraged to revisit the training setup and explore how different design choices influence the final accuracy, for example:

- the number of collocation points,
- the number of initial and boundary condition samples,
- the weighting parameter $\lambda$,
- the learning rate,
- the network depth or width,
- or the number of training epochs.

Even small changes in these settings can noticeably affect convergence speed and final solution quality.

---

### 📌 Final Remarks on PINNs as Forward Solvers

Using PINNs in the forward setting—that is, as neural solvers for differential equations—is still an active and exciting research area.

While the results in this tutorial are very promising, training PINNs can become challenging for more complex systems. In these problems, the network often relies heavily on the physics loss, since this term provides the key information about how the system evolves.

As a general rule:

- the more complex the governing physics,
- the more difficult the optimization problem,
- and the more likely training difficulties become.

Examples include:

- slow or unstable convergence,
- imbalance between different loss terms,
- difficulties in resolving sharp gradients or multiple scales,
- sensitivity to hyperparameter choices.

---

### 🔬 Current Research Topics in PINNs

A large part of the PINN literature focuses on improving exactly these issues. Popular research directions include:

- **Adaptive loss balancing.**
Automatically adjusting the trade-off between data loss and physics loss, rather than choosing λ manually.
- **Improved collocation sampling.**
Selecting collocation points in the most informative regions of the domain.
- **Residual weighting strategies.**
Reweighting individual PDE residuals to better capture steep fronts, preserve causality, or focus learning where errors are largest.
- **Advanced optimizers and training schedules.**
Combining Adam, L-BFGS, curriculum learning, or domain decomposition methods.

These are only a few examples from a rapidly growing field.

---

### 🎓 What We Hope You Learned

Across these tutorials, you have seen how PINNs can be used in several different ways:

- **Data-enhanced prediction** using observations and physics
- **Hybrid learning** where physics improves extrapolation
- **Pure forward simulation** using only equations and constraints

You have also seen that PINNs are more than just another neural network architecture—they are a flexible framework for embedding scientific knowledge into machine learning.

---

### 🚀 Looking Ahead

PINNs are especially promising for:

- inverse problems,
- parameter estimation,
- systems with scarce data,
- multiphysics applications,
- and scientific domains where traditional solvers are expensive.

Their full potential is still being explored, which makes this a particularly exciting area for future research.

---

We hope you enjoyed these tutorials and gained a practical first impression of how Physics-Informed Neural Networks work, how they are trained, and why they are such an interesting bridge between **machine learning** and **scientific computing**.